In [1]:
# To log in to huggingface
from huggingface_hub import notebook_login, login, whoami

# login()
whoami()

{'type': 'user',
 'id': '6807d07d16ba91e25220df18',
 'name': 'manify',
 'fullname': 'Manify',
 'email': 'pc2946@columbia.edu',
 'emailVerified': True,
 'canPay': False,
 'periodEnd': None,
 'isPro': False,
 'avatarUrl': '/avatars/963ae0c58505acd0128dd3058b071321.svg',
 'orgs': [],
 'auth': {'type': 'access_token',
  'accessToken': {'displayName': 'notebook2',
   'role': 'write',
   'createdAt': '2025-04-27T03:02:27.198Z'}}}

In [ ]:
"""
make_hf_datasets.py
 • loads each dataset via manify.utils.dataloaders.load
 • wraps it into a single‑row HuggingFace Dataset
 • pushes each one as its own HF repo under HF_NAMESPACE/<dataset_name>
"""

import os
import torch
from datasets import Dataset
from manify.utils.dataloaders import load

# -----------------------------------------------------------------------------
HF_NAMESPACE = os.getenv("HF_NAMESPACE", "manify")

DATASETS = {
    # distance‑based
    "cities": dict(type="distance", task=None, has_adj=False),
    "cs_phds": dict(type="distance", task="regression", has_adj=True),
    "polblogs": dict(type="distance", task="classification", has_adj=True),
    "polbooks": dict(type="distance", task="classification", has_adj=True),
    "cora": dict(type="distance", task="classification", has_adj=True),
    "citeseer": dict(type="distance", task="classification", has_adj=True),
    "karate_club": dict(type="distance", task=None, has_adj=True),
    "lesmis": dict(type="distance", task=None, has_adj=True),
    "adjnoun": dict(type="distance", task=None, has_adj=True),
    "football": dict(type="distance", task=None, has_adj=True),
    "dolphins": dict(type="distance", task=None, has_adj=True),
    # feature‑based
    "blood_cells": dict(type="feature", task="classification", has_adj=False),
    "lymphoma": dict(type="feature", task="classification", has_adj=False),
    "cifar_100": dict(type="feature", task="classification", has_adj=False),
    "mnist": dict(type="feature", task="classification", has_adj=False),
    "temperature": dict(type="feature", task="regression", has_adj=False),
    "landmasses": dict(type="feature", task="classification", has_adj=False),
    "neuron_33": dict(type="feature", task="classification", has_adj=False),
    "neuron_46": dict(type="feature", task="classification", has_adj=False),
    "traffic": dict(type="feature", task="regression", has_adj=False),
}
# -----------------------------------------------------------------------------


def to_py(x):
    if x is None:
        return []
    if isinstance(x, torch.Tensor):
        x = x.detach().cpu()
    return x.tolist()


def process(name: str) -> Dataset:
    meta = DATASETS[name]
    if meta["type"] == "distance":
        dists, labels, adj = load(name)
        feats = None
    else:
        feats, labels, adj = load(name)
        dists = None

    row = {
        "name": name,
        "type": meta["type"],
        "task": meta["task"] or "none",
        "distances": to_py(dists),
        "features": to_py(feats),
        "adjacency": to_py(adj) if meta["has_adj"] else [],
        "classification_labels": to_py(labels) if meta["task"] == "classification" else [],
        "regression_labels": to_py(labels) if meta["task"] == "regression" else [],
    }
    # wrap each value in a list → one‑row Dataset
    return Dataset.from_dict({k: [v] for k, v in row.items()})


def main():
    failed = []
    for name in DATASETS:
        print(f"→ processing {name}")
        try:
            ds = process(name)
            repo_id = f"{HF_NAMESPACE}/{name}"
            ds.push_to_hub(repo_id, private=False, num_shards=1)
            print(f"✓ pushed to https://huggingface.co/datasets/{repo_id}")
        except Exception as e:
            print(f"✗ {name} failed: {e}")
            failed.append(name)

    if failed:
        print(f"\n⚠️  {len(failed)} datasets failed:", failed)


if __name__ == "__main__":
    main()

→ processing neuron_33


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

README.md:   0%|          | 0.00/582 [00:00<?, ?B/s]

No files have been modified since last commit. Skipping to prevent empty commit.


✓ pushed to https://huggingface.co/datasets/manify/neuron_33
→ processing neuron_46


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

No files have been modified since last commit. Skipping to prevent empty commit.


✓ pushed to https://huggingface.co/datasets/manify/neuron_46
